# 03 — Fused SwiGLU / GeGLU Activations

A **compute-bound** fused kernel used in modern transformers (LLaMA, Gemma, Mistral).

### What is SwiGLU?

SwiGLU combines a gating mechanism with the SiLU activation:

```
SwiGLU(x) = SiLU(x @ W_gate) * (x @ W_up)
```

GeGLU is the same but uses GELU instead of SiLU.

### Why fuse?

Without fusion, the gate and up projections each write a full intermediate tensor to HBM:
- `gate_intermediate = x @ W_gate`  (written to HBM)
- `up_intermediate = x @ W_up`      (written to HBM)
- Read both back, apply activation, multiply

The fused kernel keeps intermediates in VMEM (on-chip SRAM), saving 2 full HBM round-trips.

In [ ]:
# ============================================================
# Colab / TPU Setup — run this cell first!
# ============================================================
# On Colab: Runtime > Change runtime type > TPU
#
# If running locally with pallas-forge already installed, skip this cell.
# ============================================================

import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "jax[tpu]", "-f",
        "https://storage.googleapis.com/jax-releases/libtpu_releases.html",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/<YOUR_USERNAME>/pallas-forge.git#egg=pallas-forge[viz]",
    ])
    print("Colab setup complete!")
else:
    print("Running locally — skipping Colab install.")

In [ ]:
import jax
import jax.numpy as jnp

from pallas_forge import fused_swiglu, fused_geglu

## 1. SwiGLU — correctness check

In [ ]:
key = jax.random.PRNGKey(42)
k1, k2, k3 = jax.random.split(key, 3)

# Typical LLaMA-style dimensions (scaled down for CPU)
batch_seq = 32   # batch * seq_len
dim = 128        # hidden dim
ffn_dim = 256    # FFN intermediate dim

x = jax.random.normal(k1, (batch_seq, dim), dtype=jnp.float32)
w_gate = jax.random.normal(k2, (dim, ffn_dim), dtype=jnp.float32)
w_up = jax.random.normal(k3, (dim, ffn_dim), dtype=jnp.float32)

# Reference: unfused
gate_ref = jax.nn.silu(jnp.matmul(x, w_gate))
up_ref = jnp.matmul(x, w_up)
expected = gate_ref * up_ref

# Fused Pallas kernel
result = fused_swiglu(x, w_gate, w_up, block_m=32, block_n=64)

max_error = float(jnp.max(jnp.abs(result - expected)))
print(f"SwiGLU: x({batch_seq}, {dim}) @ W_gate({dim}, {ffn_dim}) -> ({batch_seq}, {ffn_dim})")
print(f"Max error vs reference: {max_error:.2e}")
print(f"Correct: {bool(jnp.allclose(result, expected, atol=1e-3))}")

## 2. GeGLU — same kernel, different activation

In [ ]:
# GeGLU reference
gate_geglu_ref = jax.nn.gelu(jnp.matmul(x, w_gate))
expected_geglu = gate_geglu_ref * up_ref

# Fused
result_geglu = fused_geglu(x, w_gate, w_up, block_m=32, block_n=64)

max_error_geglu = float(jnp.max(jnp.abs(result_geglu - expected_geglu)))
print(f"GeGLU max error: {max_error_geglu:.2e}")
print(f"Correct: {bool(jnp.allclose(result_geglu, expected_geglu, atol=1e-3))}")

# Verify SwiGLU != GeGLU
print(f"\nSwiGLU and GeGLU produce different results: {not bool(jnp.allclose(result, result_geglu, atol=1e-2))}")

## 3. Batched 3D input (batch, seq_len, dim)

In [ ]:
# 3D input — batch dims are flattened internally
batch, seq_len = 2, 16
x_3d = jax.random.normal(k1, (batch, seq_len, dim), dtype=jnp.float32)

result_3d = fused_swiglu(x_3d, w_gate, w_up, block_m=32, block_n=64)

# Reference
x_flat = x_3d.reshape(-1, dim)
expected_3d = (jax.nn.silu(x_flat @ w_gate) * (x_flat @ w_up)).reshape(batch, seq_len, ffn_dim)

print(f"Input: {x_3d.shape} -> Output: {result_3d.shape}")
print(f"Correct: {bool(jnp.allclose(result_3d, expected_3d, atol=1e-3))}")

## 4. HBM traffic comparison

In [ ]:
elem_bytes = 4  # float32
M = batch_seq

# Unfused: 3 separate ops
# Op 1: gate = x @ w_gate -> write gate_intermediate
unfused_op1_read = (M * dim + dim * ffn_dim) * elem_bytes
unfused_op1_write = M * ffn_dim * elem_bytes
# Op 2: up = x @ w_up -> write up_intermediate
unfused_op2_read = (M * dim + dim * ffn_dim) * elem_bytes
unfused_op2_write = M * ffn_dim * elem_bytes
# Op 3: silu(gate) * up -> read both intermediates, write output
unfused_op3_read = M * ffn_dim * 2 * elem_bytes
unfused_op3_write = M * ffn_dim * elem_bytes
unfused_total = (unfused_op1_read + unfused_op1_write + 
                 unfused_op2_read + unfused_op2_write +
                 unfused_op3_read + unfused_op3_write)

# Fused: single kernel
fused_read = (M * dim + dim * ffn_dim * 2) * elem_bytes  # x + w_gate + w_up
fused_write = M * ffn_dim * elem_bytes  # output only
fused_total = fused_read + fused_write

savings = unfused_total - fused_total
print(f"Unfused HBM traffic: {unfused_total / 1024:.1f} KB")
print(f"Fused HBM traffic:   {fused_total / 1024:.1f} KB")
print(f"Savings:             {savings / 1024:.1f} KB ({savings / unfused_total * 100:.1f}%)")
print(f"\nThe 2 intermediate tensors ({M}x{ffn_dim} each) stay in VMEM instead of HBM.")

## 5. Different block sizes

In [ ]:
configs = [
    (16, 32),
    (32, 64),
    (32, 128),
    (32, 256),
]

print(f"{'block_m':>8} {'block_n':>8} {'max_err':>12} {'correct':>8}")
print("-" * 44)

for bm, bn in configs:
    r = fused_swiglu(x, w_gate, w_up, block_m=bm, block_n=bn)
    err = float(jnp.max(jnp.abs(r - expected)))
    ok = bool(jnp.allclose(r, expected, atol=1e-3))
    print(f"{bm:>8} {bn:>8} {err:>12.2e} {str(ok):>8}")

## Key takeaways

1. **SwiGLU/GeGLU** are standard in modern LLMs (LLaMA, Gemma, Mistral)
2. **Fusion eliminates intermediate tensors** — gate and up projections stay in VMEM
3. This is a **compute-bound** kernel (dominated by the two matmuls), unlike the memory-bound RMSNorm
4. On TPU, block sizes determine MXU utilization — too small and the systolic array is starved

Next: [04_auto_tuning.ipynb](04_auto_tuning.ipynb) — systematic auto-tuning with heatmaps.